# 3 · (Optional) Add IMERG to your own observation table and score it

If you have rain-gauge totals for the same stations and periods, this notebook adds an **IMERG**
column to your table (right after the column you choose) and prints common verification scores.

Your table needs a station-ID column and a time column holding the **end** of each accumulation
period. See `examples/observations_template.csv` for the expected layout.

In [1]:
# Cell 1 – settings
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # so the notebook can import imerg_utils.py from the repo root
import pandas as pd
import imerg_utils as iu

OBS_CSV        = '../examples/observations_template.csv'  # replace with your own file
IMERG_CSV      = '../output/imerg_6h_demo_2024_07_19.csv' # written by notebook 2
OUT_CSV        = '../output/observations_with_imerg.csv'

OBS_STATION_COL = 'station_id'
OBS_TIME_COL    = 'datetime_end'
OBS_TIME_FORMAT = '%Y-%m-%d %H:%M'   # e.g. '%Y:%m:%d-%H:%M:%S' for 2024:07:19-06:00:00
OBS_UTC_OFFSET  = 0                  # 6 if your times are Bangladesh time
OBS_VALUE_COL   = 'observed_mm'
WET_MM          = 1.0                # rain / no-rain threshold for POD, FAR, CSI

In [2]:
# Cell 2 – merge
obs   = pd.read_csv(OBS_CSV)
imerg = pd.read_csv(IMERG_CSV, parse_dates=['datetime_end_utc'])
merged = iu.merge_into_table(obs, imerg, OBS_STATION_COL, OBS_TIME_COL, OBS_TIME_FORMAT,
                             after_col=OBS_VALUE_COL, new_col='IMERG', utc_offset_hours=OBS_UTC_OFFSET)
merged.to_csv(OUT_CSV, index=False)
print(f'{len(merged)} rows -> {OUT_CSV};  {merged.IMERG.isna().sum()} rows without an IMERG value')
merged.head()

0 rows -> ../output/observations_with_imerg.csv;  0 rows without an IMERG value


,station_id,name,datetime_end,observed_mm,IMERG


In [3]:
# Cell 3 – scores of IMERG (and any other numeric columns you list) against the observations
compare = ['IMERG']   # e.g. ['IMERG', 'model_a', 'model_b']
pd.DataFrame({c: iu.scores(merged[OBS_VALUE_COL], merged[c], WET_MM) for c in compare}).T

,n,pearson_r,rmse_mm,mae_mm,bias_ratio,hits,misses,false_alarms,pod,far,csi
IMERG,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
